# Function Calling
- Function Calling은 OpenAI API에서 특정 함수 정의(JSON schema 기반)를 주고, 모델이 해당 함수를 호출하는 형식의 응답을 생성하게 하는 기능입니다.
- 챗봇, 에이전트(Agent), 도구(Tool) 사용 등 다양한 자동화 및 플러그인 연동에 활용됩니다.

- 지원 모델 gpt-3-turbo, gpt-4, gpt-4-turbo  (2025.1 현재)

- 모델이 우리의 코드를 호출하도록 해서 우리의 함수들을 모델이 호출할수 있도록 하거나
- 모델이 우리가 원하는 특정 모양과 형식의 output 을 갖도록 강제할수 있다.

- https://platform.openai.com/docs/guides/function-calling?api-mode=chat

![](https://cdn.thenewstack.io/media/2024/05/5026cffa-rag_function_call-1024x835.jpeg)


In [1]:
import os 

from dotenv import load_dotenv
load_dotenv()

print(f'OPENAI_API_KEY={os.getenv("OPENAI_API_KEY")[:20]}...')

OPENAI_API_KEY=sk-proj-iKU13YeoxNgF...


In [2]:
from langchain_openai.chat_models.base import ChatOpenAI
from langchain_core.prompts.prompt import PromptTemplate

In [4]:
llm = ChatOpenAI(temperature=0.1)

prompt = PromptTemplate.from_template("Who is the weather in {city}")

chain = prompt | llm

response = chain.invoke({'city': 'rome'})

print(response.content)

I'm sorry, I am not able to provide real-time weather updates. I recommend checking a reliable weather website or app for the most up-to-date information on the weather in Rome.


In [5]:
# 사전학습 모델은 '실시간 정보' 불가.

# 하지만!  우리에게 실시간 데이터를 가져오는 함수가 있다고 가정해보자.

In [6]:
# 필요한 정보는 위도, 경도 좌표라 하자.
def get_weather(lon, lat):
    print(f'🟨날씨 API 호출 lon:{lon}, lat:{lat}')

In [ ]:
# 우리가 'Function calling'로 할 수 있는 일은 앞선 실망스러운(?) 응답을 받는 대신에 
# gpt-3, gpt-4 에게 우리에게 '이러이런한 일'을 해주는 get_weather() 라는 함수가 있다고 말해주는 거다.
# 그러면, gpt 는 우리의 질문을 보게 되겠고, 함수를 사용하는 것이 이런 실망스러운 답변을 주는 것 보다 나은지 판별할거다.

# 어떻게 GPT 나 LLM 을 그렇게 동작하도록 만들수 있을까?
# LLM 이 get_weather 라는 함수가 있다는 걸 알도록 할수 있을까?

# 생각보다 쉽다!
# 이 함수를 위한 JSON schema 를 만드는 거다! ↓↓

# 함수의 스키마
JSON schema 로 정의

In [7]:
# 함수의 작동방식과 필요한 것들을 설명해주는 것이 스키마
function_schema = {
    "name": "get_weather",   # 함수의 이름
    # description : 함수가 무슨 일을 하는지 기술
    # ↓'위도와 경도를 받아서 특정장소의 날씨정보를 가져오는 함수"
    "description": "function that takes longitude and latitude to find the weather of a place",  

    # 파라미터 기술
    "parameters": {
        "type": "object", 
        "properties": {
            "lon": {"type": "string", "description": "The longitude coordinate"},
            "lat": {"type": "string", "description": "The latitude coordinate"},
        },
    },

    # 필수사항 기술
    "required": ["lon", 'lat'],
}

# ChatOpenAI 에 functions 넣기

In [9]:
llm = ChatOpenAI(
    temperature=0.1,
).bind(   # <- ChatOpenAI 에게 전달할 인수를 추가
    functions=[   # 원하는 만큼 많은 함수 전달 가능.
        function_schema,  # 준비한 함수 스키마 전달.
    ],

    # AI 에게 선택권을 주어서 AI가 필요에 따라 선택하여 사용하게 할수 있다.
    function_call='auto',
)

chain = prompt | llm
response = chain.invoke({"city": 'rome'})
print(response.content)  # 내용 없다.

response  # 과연?

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"lon":"12.4964","lat":"41.9028"}', 'name': 'get_weather'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 74, 'total_tokens': 98, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DcsQAXE1wczEhL6R58tpAUw3184bC', 'service_tier': 'default', 'finish_reason': 'function_call', 'logprobs': None}, id='lc_run--019e027a-45bc-7c42-971b-1abb123b4804-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 74, 'output_tokens': 24, 'total_tokens': 98, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [ ]:
"""
AIMessage(
    content='',   # content 없이 아래의 function calling 이 응답되었다.
    additional_kwargs={

        # ↓ 함수가 호출되었다!  
        # 모델은 get_weather 라는 함수를 호출해 줬으면 해!  라고 말하고 있는거다.
        # 그리고, "네가 경도와 위도에 대한 argument 를 제공해 줬으면 해" 라고 하고 있는 거다.        
        # rome 의 lon 과 lat 값을 전달해 주었다!  오오오오!

    
        'function_call': {
            'arguments': '{"lon":"12.4964","lat":"41.9028"}', 
            'name': 'get_weather'
        }, 
        'refusal': None},

"""

## 